# Hockey

In [ ]:
#| include: false

import warnings
warnings.filterwarnings('ignore')

"You miss 100 percent of the shots you don't take." -- Wayne Gretzky, a.k.a. The Great One

"There really is nothing more exhausting and quite frankly more stupid than the 'analytics' debate in hockey. The debate is almost always started by those who don't like or trust 'analytics', and these people frame it like both sides are arguing - at this point they really aren't." -- Josh Younggren, owner of Evolving Hockey.

"You may have heard what I call the 'underpants' rule: if you can replace the word 'analytics' with 'underpants' and the statement is equally sound, check your logic." -- Andrew C. Thomas, hockey analyst.

## Python Modules

We will use the following Python modules in this chapter that we have used in previous chapters:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

We will also use the following modules in this chapter:

In [ ]:
from nhlpy import NHLClient
from sportypy.surfaces.hockey import NHLRink
import time

The `nhlpy` module (installed as `nhl-api-py`) is a Python wrapper for the NHL's official API. It provides access to team rosters, player statistics, game schedules, play-by-play data, and much more. The `NHLRink` class from `sportypy` allows us to draw an NHL rink as a background for shot charts, just as we used `NBACourt` in the basketball chapter.

::: {.callout-note}
The `nhl-api-py` and `sportypy` packages are not installed by default. Before running this chapter, install them with:

```bash
pip install nhl-api-py sportypy
```
:::

In [ ]:
#| echo: false
#| eval: true

sns.set_theme(style="whitegrid")

The textbook's `isapy_data` module provides the `nhl_data_hockey_reference` dataset, which we use in the first part of this chapter. For current roster and statistics data, we access the NHL's official API directly, as described later.

In [ ]:
import isapy_data

## If you don't know anything about hockey

For the purposes of this book, "Hockey" will refer to Ice Hockey, the game played in the Winter Olympics, as opposed to Hockey or Field Hockey, the game played in the Summer Olympics. Many of the analytic principles may well apply to both, but the authors have not studied Field Hockey.

In [ ]:
#| fig-cap: "Hockey Rink"
#| label: fig-rink
#| echo: false

from IPython.display import Image, display
display(Image(filename="fig/NHL-Hockey-Rink-Diagram.jpg"))

Hockey belongs to a family of sports where the objective is for a team of players to move an object from one end of a field or court to the other end and ultimately deposit it into a goal of some sort. While hockey is most similar to soccer in these regards, it also has elements of basketball.

In hockey, each team consists of 5 skaters and one goaltender, who skate on an ice rink approximately 200 feet (60 meters) long and 85-100 feet (25-30 meters) wide. The skaters are divided into offensive players (Left Wing, Center, and Right Wing) and two Defenders, who are generally not classified into left or right. Each player carries a hockey stick, which is used to move the puck, a hard rubber disk about 3 inches (7.5 cm) in diameter, 1 inch (2.5 cm) thick, and weighing about 6 ounces (165 grams), along the ice. The sides of the rink are hemmed in by "boards" about 3 feet (1 meter) high, and Plexiglas about another 6 feet (1.8 meters high), so in general the puck does not go out of play with nearly the frequency of a soccer ball. A puck would have to be shot quite high to go out of play and/or into the seats with the spectators, who watch the game through the clear Plexiglas if they are close to the rink. The boards and glass allow spectators to be very close to the players, easily able to touch them if it weren't for the glass.

The object is to hit the puck with the stick into the net, a 6 feet (1.8 meter) wide by 4 feet (1.2 meter high) target. The net is guarded by a goaltender, a player on the team whose job is primarily to keep the puck out of the net. The goalie, as he or she is colloquially known, wears many extra thick pads and face masks to keep from getting hurt by a puck that can travel at over 100 miles per hour (160 kilometers per hour) off of the sticks of players with the hardest shots. The net is anchored near each end of the ice, but unlike soccer or water polo there is room to play behind the net. This requires goalies to have a neck like an owl as they must look behind them even though the opening to the net is only in front of them.

Along with the skaters in play, there are approximately 15 players in reserve at any given time. Players may be substituted freely in and out of the game, and it is rare for a skater to play more than 90 seconds before going to the bench for a reserve skater. With skaters traveling sometimes more than 30 miles per hour (50 kilometers per hour), they tire quickly. It is quite rare for a skater to play more than 30 minutes of a normal 60 minute game (three 20-minute periods), with 19-25 minutes being the average time on ice. There is no stoppage in play for substitutes to come on and off the ice --- players jump over the boards in front of their bench (where there is no Plexiglas) or exit through a door that opens in front of the bench. Hockey is full of action.

In [ ]:
#| fig-cap: "Substitution in hockey"
#| label: fig-shift-change
#| echo: false

display(Image(filename="fig/shift-change.jpg"))

If a player commits a foul, such as hitting another player with the stick, tripping another player, or interfering with the fair course of action, it is called a penalty and that player is sent to the penalty box for a period of time. While that player is in the penalty box, his or her team must play with one less player than the other team. The offending player's team is said to be shorthanded, while the other team with one more skater on the ice is said to be on the power play. Most penalties are considered minor penalties and the player is sent off for 2 minutes, but for dangerous plays like fighting or hitting someone even accidentally on or near the head with your stick it is called a major penalty and the player is sent off for 5 minutes. Five minutes is quite a long time to play with one less skater than your opponent, and it quite often results in giving up one or more goals to the opponents.

Much like other chapters, this is just a basic introduction. For more details, you can read about ice hockey at [https://www.rookieroad.com/ice-hockey/what-is/](https://www.rookieroad.com/ice-hockey/what-is/).

## What you will need to complete this chapter

### Hockey Reference {.unnumbered}

You may remember from our chapter on web scraping that we used data from Hockey Reference. The `nhl_data_hockey_reference` dataset bundled with this textbook's `isapy_data` module was scraped from Hockey Reference using the techniques covered in that chapter. We will use this dataset extensively in the first part of this chapter.

### Moneypuck {.unnumbered}

There is a great website that makes a ton of hockey data available for download called Moneypuck. Almost all of their data is available as a CSV file. You can download this data using the links given on their download page. For example, here is how you could get all of the skater (i.e., non-goalie) information for the 2024-2025 season.

In [ ]:
#| eval: false

year = "2025"
link = f"http://moneypuck.com/moneypuck/playerData/seasonSummary/{year}/regular/skaters.csv"
df = pd.read_csv(link)

Here are a few more links:

1. All game data going back to the 2008-9 season
    * [http://moneypuck.com/moneypuck/playerData/careers/gameByGame/all_teams.csv](http://moneypuck.com/moneypuck/playerData/careers/gameByGame/all_teams.csv)
2. Shots (you can iterate over years)
    * [http://peter-tanner.com/moneypuck/downloads/shots_2019.zip](http://peter-tanner.com/moneypuck/downloads/shots_2019.zip)

### NHL API via nhl-api-py {.unnumbered}

The NHL provides an official API that we can access through the `nhl-api-py` Python package. This package wraps the API's endpoints into a clean Python interface through the `NHLClient` class. We will use it later in this chapter to access team rosters, player statistics, and play-by-play data.

In [ ]:
client = NHLClient()

## Exploring Ice Hockey through Basic and Advanced Statistics

The data set that we use in the initial part of this chapter was obtained by scraping Hockey Reference using the techniques from the web scraping chapter. The data set has 12,375 observations related to players from the 2011-12 season through the 2022-23 season. The `isapy_data` module provides this data as `nhl_data_hockey_reference`.

In [ ]:
nhl_data = isapy_data.load_data("nhl_data_hockey_reference")
print(f"Shape: {nhl_data.shape}")
nhl_data.head()

The dataset contains 48 columns spanning basic counting statistics (goals, assists, points), shot metrics (shots, shooting percentage), and advanced analytics (Corsi, Fenwick, PDO, zone starts). To help refresh your memory, you could explore the data. Some of these columns may seem quite obvious, but others might seem difficult to understand for the non-avid hockey fan.

Note that `age` is stored as a decimal (e.g., `24.0`) because two players are missing an age, and pandas converts an integer column with missing values to floating point. Hockey Reference records age as a whole number (a player's age on February 1 of that season), so we convert it to pandas' nullable integer type, `Int64`, which allows missing values.

In [ ]:
nhl_data['age'] = nhl_data['age'].astype('Int64')
nhl_data[['player', 'pos', 'tm', 'age', 'gp', 'g', 'a', 'pts',
          'toi', 'cf_percent', 'ff_percent', 'season']].head(10)

## *Corsi* and *Fenwick* metrics

Amongst the first advanced analytics to be used in hockey to measure player effectiveness are Corsi and Fenwick. For over 100 years basic metrics such as goals scored (or allowed for goaltenders), assists, and penalty minutes were the primary metrics used to measure player effectiveness. Plus-minus also evolved as a metric, where a player's plus-minus rating was equal to the number of goals scored by their team while they were on the ice less the number of goals allowed by their team when that player was on the ice. This helped, but its flaw is that the metric often depends on how good a player's teammates are. A very low level player might have an excellent plus-minus rating because he happens to be on the ice at the same time as a scoring superstar like Auston Matthews or Sidney Crosby.

Although not terribly advanced, Corsi and Fenwick are the next level in doing more than simply counting. Corsi is the difference in shot attempts (shots on goal + missed shots + blocked shots) when the player is on the ice and shot attempts by his opponent when he's on the ice. Note that only 5-on-5 scenarios are counted. Another measure, Fenwick, is similar but does not include blocked shots.

In [ ]:
#| fig-cap: "Nathan MacKinnon"
#| label: fig-mackinnon
#| echo: false

display(Image(filename="fig/nathan_mackinnon.jpg"))

Though we describe Corsi (Fenwick) as a difference, it may help to view them as percentages. For example, as a measure of effectiveness, we can see whether a player being on the ice is not just effective as a raw Corsi number but as a percentage of time on the ice. To do this, we can look at Corsi as a percentage of the Corsi "for" (SOG + missed shots + blocked shots for that player's team) while a player is on the ice, divided by the Corsi "for" plus the Corsi "against" (SOG + missed shots + blocked shots by the opponent while that player is on the ice). We can then plot Corsi percent by year below for one player, Nathan MacKinnon of the Colorado Avalanche:

In [ ]:
mac = nhl_data[nhl_data['player'] == 'Nathan MacKinnon'].copy()

In [ ]:
#| label: fig-mac-corsi
#| fig-cap: "Corsi percentage values (y) by age (x) for Nathan MacKinnon of the Colorado Avalanche."
#| fig-width: 5

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(mac['age'], mac['cf_percent'] / 100, marker='o')
ax.set_xlabel('Age')
ax.set_ylabel('Corsi Percentage')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.set_xticks(range(18, 29))
plt.tight_layout()
plt.show()

As we see in the graph, as MacKinnon has gained experience in the NHL, his effectiveness on the ice has also increased. This graph indicates that not only are the scoring chances when he is on the ice more in favor of his team, but that they are increasingly in favor of his team as he has aged and become more proficient in the NHL game.

Corsi and Fenwick calculations are possible today with effective shot tracking which began in 2007-2008. As with all advanced metrics, it does make it difficult to compare more modern players with past players who played before this data was collected. This means not just for players of lore like Maurice Richard, Johnny Bucyk, or Phil Esposito, but even Wayne Gretzky and Mario Lemieux.

## Time On Ice (TOI) by Positions

Earlier we mentioned that it is highly unusual for a skater to even play half the game. 
Goalkeepers generally play the entire game, but we will exclude them from this analysis.
If we are managing a team, whether at the highest professional level or in single-digit aged youth, we want to manage the players' time on the ice so that they do not become overly fatigued, whether in the waning minutes of a game or after several games in close succession with few or no days off. 
To understand this, we should examine the time spent on the ice. 
First we should understand the differences by position played. 
As NHL teams generally carry 4 offensive lines (LW, C, RW) on the active roster for a game but only 3 pairs of defenders, we might expect that defenders are playing more minutes throughout a game and throughout the season.
So first, we can create a graphical distribution of TOI by position, first together and then in individual facets.
To do this, we will take our NHL data and filter it by position, eliminating goalkeepers. 
We will also filter out anyone with a TOI less than 200 minutes for the season, as that would be indicative of a bit player that played in perhaps a few games out of an 82 game season, and then create density plots.
We are also filtering out anyone with a generic designation of “Wing” or “Forward”, as these positions were added to stats in 2018 and 2020, respectively.

In [ ]:
#| label: fig-toi
#| fig-cap: "Density plots for Time on Ice by position."
#| fig-width: 5

toi_data = nhl_data[
    (nhl_data['toi'] >= 200) &
    (~nhl_data['pos'].isin(['F', 'W']))
].copy()

g = sns.FacetGrid(toi_data, col='pos', col_wrap=2, height=4, aspect=1.3)
g.map(sns.kdeplot, 'toi', fill=True, color='gray', alpha=0.25)
g.set_axis_labels('Time on Ice (min)', 'Density')
g.set_titles('{col_name}')
plt.tight_layout()
plt.show()

As we view the graphs, we see the Center, Left Wing, and Right Wing graphs look relatively similar. This is to be expected (as mentioned above) as normally each team will group a center, left wing, and right wing as a "line" and they will play together, as they know each other's strengths and weaknesses and tendencies. Having three defensive pairs rather than four offensive lines means that defensemen tend to play more minutes on the ice each game.

## Evaluating Players

As we mentioned earlier, Corsi percent is a newer metric for evaluating players. What if we wanted to see who had the maximum Corsi per year by position? We can take the NHL data that we've been working with in this chapter, and then:

* Filter out any player that didn't play at least 200 minutes,
* Group the players by season and position, and
* List the maximum.

In [ ]:
top_cor = (nhl_data[nhl_data['toi'] >= 200]
           .groupby(['season', 'pos'])
           .agg(max_corsi=('cf_percent', 'max'))
           .reset_index()
           .sort_values('max_corsi', ascending=False, ignore_index=True))
top_cor.head()

The unlabeled first column is the DataFrame's index (pandas' row labels), not a player attribute. Setting `ignore_index=True` re-numbers the rows from 0 after sorting.

The above DataFrame does not give us the entire dataset associated with this entry; it's only the season, position, and max Corsi, but not the player's name. How can we find that? Consider the following code.

In [ ]:
nhl_filtered = nhl_data[nhl_data['toi'] >= 200]
corsi_idx = nhl_filtered.groupby(['season', 'pos'])['cf_percent'].idxmax()
top_cor_players = (nhl_filtered.loc[corsi_idx]
                   [['season', 'player', 'pos', 'tm', 'cf_percent']]
                   .sort_values('cf_percent', ascending=False, ignore_index=True))
top_cor_players.head()

In [ ]:
#| label: fig-corsi-pos
#| fig-cap: "Average Corsi values over time by several positions."
#| fig-width: 5

nhl_tmp = (nhl_data[nhl_data['toi'] >= 200]
           .groupby(['season', 'pos'])
           .agg(avg_corsi=('cf_percent', 'mean'))
           .reset_index())

fig, ax = plt.subplots(figsize=(10, 6))
palette = sns.color_palette("Dark2")
for i, pos in enumerate(nhl_tmp['pos'].unique()):
    pos_data = nhl_tmp[nhl_tmp['pos'] == pos]
    ax.plot(pos_data['season'], pos_data['avg_corsi'],
            marker='o', label=pos, color=palette[i % len(palette)])
ax.set_xlabel('Season')
ax.set_ylabel('Average Corsi')
ax.set_xticks(range(2012, 2024))
ax.tick_params(axis='x', rotation=45)
ax.legend(title='Position')
plt.tight_layout()
plt.show()

You can see here that the data are a bit confusing. It appears that starting in 2018, the position "W" was added, and in 2020, the position "F" also appears. These are abbreviations for Wing and Forward, respectively, and they add some spice to the visualization. More on this later.

___

## Research Block: Fighting in the NHL

In the past, ice hockey was known for tolerating to some degree fighting between the players. When a particular player might have felt that some injustice occurred on the ice, such as a player being hit from behind or otherwise hit dangerously or needlessly, a fight might occur between the players. Increasingly, teams would have players who might have had lesser technical skills than other skaters, but were the ones that would be most likely to get into a fight. These skaters sometimes had nicknames like the enforcers, goons, or designated fighters. The result of the fight was usually that there would be some mild entertainment for the fans for a minute or two, the fighters would be sent to the penalty box for five minutes, and the game would resume.

In [ ]:
#| fig-cap: "Hockey fight"
#| label: fig-fight
#| echo: false

display(Image(filename="fig/hockey_fight.jpg"))

Rules have changed throughout the years to limit the impact and frequency of fights, particularly as more serious acute injuries and chronic conditions like chronic traumatic encephalopathy (CTE) have led to much more than a few bruises or scars as the results of fights. Nevertheless fighting itself has never been eliminated, particularly at the highest professional levels. Defenders of the practice have said this is how the players police the game themselves. They claim that without fighting that it would enable players to take cheap shots at opposing players and fear nothing more than a 2 minute minor penalty. They also claim that it can be used to change the momentum of a game, for example to inspire a team that is losing. Opponents of the practice state that fighting turns a beautiful game like hockey and turns it into a medieval practice where skill is diminished by brawn and that what would be called felonious assault if it happened on the concourses of the arena is somehow turned into just a part of the game, and as such they feel like they can't have their children watching the game. The NHL remains the only major hockey league where fighting is considered a part of the game as opposed to an automatic ejection and suspension from further competition.

While the authors here take no position on the role of fighting in the game, other researchers have examined different aspects of fighting and hockey. In the manuscript "Do not drop your gloves: 'Winning' a fight in the National Hockey League does not improve chances to win the game", available at [https://www.sciencedirect.com/science/article/abs/pii/S146902922100159X](https://www.sciencedirect.com/science/article/abs/pii/S146902922100159X) and published in the journal Psychology of Sport and Exercise, Nadav Goldschmied and Michael Apostol studied NHL games in which fights have occurred and if there was a change in momentum. As many fights in an NHL game occur when the game is out of hand, such as a 5-1 score late in the final period, the researchers instead only viewed games where the score was tied or there was a one-goal difference. They also had to determine which player "won" the fight. Of course the NHL does not officially judge a winner or a loser of a fight nor award different penalty minutes to the winner or the loser, so instead they had to rely on online voting by fans as to who won the fight, and used objective data from the fight to validate the fans' opinions.

The researchers found that the team with the winning player was no more likely to win the game in a tied game than the losing player's team. Furthermore, when a team was down one goal, they also found no real momentum effect that would cause the team trailing at the time to change the game's outcome. What was not studied, however, was its effect later in the year, particularly as those two teams or players meet again.

___

## Accessing the NHL API

Rather than relying on bundled snapshots of historical data, we access the NHL's official API directly to retrieve current rosters, statistics, and play-by-play data. The NHL updated their API before the 2023-24 season, and the `nhl-api-py` Python package provides a modern wrapper for the new API.

The `NHLClient` class is the main entry point. We created an instance earlier in the chapter:

In [ ]:
#| eval: false

from nhlpy import NHLClient
client = NHLClient()

### Teams and Rosters {.unnumbered}

We can retrieve the Colorado Avalanche roster for the 2024-25 season using the `team_roster()` method. The result is a dictionary with three keys: `'forwards'`, `'defensemen'`, and `'goalies'`.

In [ ]:
#| cache: true

roster_data = client.teams.team_roster('COL', '20242025')

# Combine all position groups into a single DataFrame
rows = []
for group in ['forwards', 'defensemen', 'goalies']:
    for player in roster_data.get(group, []):
        rows.append({
            'id': player['id'],
            'name': f"{player['firstName']['default']} {player['lastName']['default']}",
            'number': player.get('sweaterNumber', None),
            'position': player['positionCode'],
            'shoots': player.get('shootsCatches', None),
        })

avs_roster_api = pd.DataFrame(rows)
print(f"Roster size: {len(avs_roster_api)}")
avs_roster_api.head(10)

### Team Statistics {.unnumbered}

The `stats` module provides league-wide team statistics. We can retrieve the current season's team summary and convert it to a DataFrame.

In [ ]:
#| cache: true

team_stats = client.stats.team_summary(
    start_season='20242025',
    end_season='20242025'
)
team_stats_df = pd.DataFrame(team_stats)
team_stats_df[['teamFullName', 'gamesPlayed', 'wins', 'losses',
               'points', 'goalsFor', 'goalsAgainst']].head(10)

### Skater Statistics {.unnumbered}

We can also retrieve individual skater statistics across the league. The `skater_stats_summary()` method returns a list of dictionaries that we can convert to a DataFrame.

In [ ]:
#| cache: true

skaters = client.stats.skater_stats_summary(
    start_season='20242025',
    end_season='20242025',
    limit=100
)
skaters_df = pd.DataFrame(skaters)
skaters_df[['skaterFullName', 'teamAbbrevs', 'gamesPlayed',
            'goals', 'assists', 'points', 'plusMinus']].head(10)

### Play-by-Play Data {.unnumbered}

One of the most powerful features of the NHL API is access to play-by-play data for individual games. Each play includes event type, time, period, and --- for shot events --- the x and y coordinates on the rink.

To collect play-by-play data for the Colorado Avalanche's 2024-25 season, we first retrieve the schedule to get game IDs, then loop over each game to fetch the plays.

In [ ]:
#| cache: true

schedule = client.schedule.team_season_schedule('COL', '20242025')
games = schedule.get('games', [])

# Filter to regular season games (gameType == 2)
reg_games = [g for g in games if g.get('gameType') == 2]
print(f"Regular season games: {len(reg_games)}")
print(f"First game ID: {reg_games[0]['id']}")

In [ ]:
#| cache: true

all_plays = []
for i, game in enumerate(reg_games):
    game_id = game['id']
    pbp = client.game_center.play_by_play(game_id)
    for play in pbp.get('plays', []):
        details = play.get('details', {})
        if 'xCoord' in details:
            all_plays.append({
                'game_id': game_id,
                'game_type': game.get('gameType'),
                'period': play['periodDescriptor']['number'],
                'time': play.get('timeInPeriod'),
                'event': play.get('typeDescKey'),
                'x': details['xCoord'],
                'y': details['yCoord'],
                'zone': details.get('zoneCode'),
                'shot_type': details.get('shotType'),
                'team_id': details.get('eventOwnerTeamId'),
            })
    if (i + 1) % 20 == 0:
        print(f"  Processed {i + 1}/{len(reg_games)} games...")
    time.sleep(0.5)

pbp_df = pd.DataFrame(all_plays)
print(f"\nTotal plays with coordinates: {len(pbp_df)}")
print(f"Unique events: {pbp_df['event'].unique()}")

### Shot Charts {.unnumbered}

One point that helps us inform how teams are scoring goals is the location from where they are shooting. The NHL records this data using player tracking systems available in all arenas, and we can access this information through the play-by-play data we just collected. Furthermore, we can visualize this data by plotting it on an NHL rink background using the `sportypy` package. This will give us similar results as the shot charts in the basketball chapter.

We need the Colorado Avalanche's team ID to filter the data. The Avalanche's team ID is 21 in the NHL API.

In [ ]:
avs_team_id = 21
avs_goals = pbp_df[
    (pbp_df['event'] == 'goal') &
    (pbp_df['team_id'] == avs_team_id)
].copy()
print(f"Avalanche goals: {len(avs_goals)}")

Because the Avalanche attack different ends in different periods, the raw x-coordinates can be positive or negative depending on which direction they are skating. We normalize all goals to the offensive zone by taking the absolute value of x, so every shot appears on the same half of the rink.

In [ ]:
#| label: fig-nhl-avs-goals
#| fig-cap: "All goal locations for the Colorado Avalanche during the 2024-25 regular season, normalized to the offensive zone."
#| fig-width: 5

avs_goals = avs_goals.copy()
avs_goals['x_norm'] = avs_goals['x'].abs()

color_updates = {
    'plot_background': '#ffffff',
    'boards': '#000000',
    'ozone_ice': '#ffffff',
    'nzone_ice': '#ffffff',
    'dzone_ice': '#ffffff',
    'center_line': '#c8102e',
    'zone_line': '#0033a0',
    'goal_line': '#c8102e',
}

rink = NHLRink(color_updates=color_updates)
ax = rink.draw(display_range='ozone')
ax.scatter(avs_goals['x_norm'], avs_goals['y'],
           alpha=0.5, s=30, color='#6F263D', zorder=100)
ax.set_title('')
plt.show()

Suppose we are interested in just the goals scored in the postseason. The play-by-play data we collected includes only regular season games. To get playoff data, we can filter the schedule for playoff games (`gameType == 3`) and fetch their play-by-play data as well.

In [ ]:
#| cache: true

playoff_games = [g for g in games if g.get('gameType') == 3]
print(f"Playoff games: {len(playoff_games)}")

playoff_plays = []
for game in playoff_games:
    game_id = game['id']
    pbp = client.game_center.play_by_play(game_id)
    for play in pbp.get('plays', []):
        details = play.get('details', {})
        if 'xCoord' in details:
            playoff_plays.append({
                'game_id': game_id,
                'period': play['periodDescriptor']['number'],
                'event': play.get('typeDescKey'),
                'x': details['xCoord'],
                'y': details['yCoord'],
                'shot_type': details.get('shotType'),
                'team_id': details.get('eventOwnerTeamId'),
            })
    time.sleep(0.5)

if playoff_plays:
    playoff_df = pd.DataFrame(playoff_plays)
    avs_playoff_goals = playoff_df[
        (playoff_df['event'] == 'goal') &
        (playoff_df['team_id'] == avs_team_id)
    ]
    print(f"Avalanche playoff goals: {len(avs_playoff_goals)}")
else:
    print("No playoff games found for this season.")

In [ ]:
#| label: fig-nhl-avs-playoff-goals
#| fig-cap: "All goal locations for the Colorado Avalanche in playoff games from the 2024-25 season."
#| fig-width: 5

if playoff_plays and len(avs_playoff_goals) > 0:
    rink = NHLRink(color_updates=color_updates)
    ax = rink.draw(display_range='full')
    ax.scatter(avs_playoff_goals['x'], avs_playoff_goals['y'],
               alpha=0.8, s=40, color='#6F263D', zorder=100)
    ax.set_title('')
    plt.show()
else:
    print("No playoff goal data available to plot.")

### Expected Goals {.unnumbered}

The shot location data above is also the foundation of expected goals (xG), and is one of the most widely used advanced metrics. 
Rather than treating every shot attempt equally (like Corsi and Fenwick), an xG model assigns each shot a probability of resulting in a goal.
Most xG models typically rely on features such distance and angle to the net, shot type (wrist shot, slap shot, one-timer), whether it came off a rebound or a rush, and the game situation.  
Summing these probabilities over a game or season gives a team or player an expected goals total, which can then be compared to actual goals scored to identify who is over- or under-performing relative to the quality of chances they're generating or allowing.
Can you think of additional features that could be used in an xG model?

Interested readers can easily fit an xG model using logistic regression (or your favorite ML model) on historical shot data, e.g. the `pbp_df` DataFrame above.
Recall that this data set has `x`, `y`, `shot_type`, and `event` columns.

## Lab Seven: Interactive Dashboards {#sec-dashboards}

You are now well into this book, and your instructor has certainly given you assignments to turn in by now. You may be wondering how to show your work effectively to people who do not know how to code in Python or even care about coding. A useful tool for displaying complex, potentially high-dimensional data sets and the information they contain is to create an interactive dashboard for data exploration.

In R, the standard tool for building interactive dashboards is Shiny. In Python, there are several excellent options, each with different strengths. We briefly describe three of the most popular frameworks below.

Each example below is a self-contained script meant to be saved and run from the terminal — not executed inside this notebook. Each script includes its own `import` statements at the top, as is standard for standalone Python files.

### Streamlit {.unnumbered}

[Streamlit](https://streamlit.io) is the most popular framework for building data applications in Python. It is designed to be simple and fast --- you write a plain Python script and Streamlit turns it into a web app. Streamlit's API is intuitive: `st.selectbox()` creates a dropdown, `st.plotly_chart()` renders an interactive plot, and the app automatically re-runs when the user changes an input.

In [ ]:
#| eval: false

# streamlit_app.py
import streamlit as st
import plotly.express as px
import pandas as pd
from nhlpy import NHLClient

client = NHLClient()
team_stats = client.stats.team_summary(
    start_season='20242025', end_season='20242025'
)
df = pd.DataFrame(team_stats)

exclude = ['teamFullName', 'teamId', 'seasonId', 'ties']
stat_cols = [c for c in df.columns if c not in exclude]

st.title("NHL Team Statistics Explorer")
x_var = st.selectbox("X Variable", stat_cols,
                     index=stat_cols.index('wins'))
y_var = st.selectbox("Y Variable", stat_cols,
                     index=stat_cols.index('losses'))

fig = px.scatter(df, x=x_var, y=y_var,
                 hover_name='teamFullName')
st.plotly_chart(fig)

You would save this as `streamlit_app.py` and run it with `streamlit run streamlit_app.py` from the terminal.

### Dash {.unnumbered}

[Dash](https://dash.plotly.com) is built by the same team behind Plotly. It gives you more control over the application layout and uses a callback-based architecture that is closer to R's Shiny. Dash applications have two main components: the layout (HTML structure) and callbacks (Python functions that update the display when inputs change).

In [ ]:
#| eval: false

# dash_app.py
from dash import Dash, dcc, html, Input, Output
import plotly.express as px
import pandas as pd
from nhlpy import NHLClient

client = NHLClient()
team_stats = client.stats.team_summary(
    start_season='20242025', end_season='20242025'
)
df = pd.DataFrame(team_stats)

exclude = ['teamFullName', 'teamId', 'seasonId', 'ties']
stat_cols = [c for c in df.columns if c not in exclude]

app = Dash(__name__)
app.layout = html.Div([
    html.H1("NHL Team Statistics Explorer"),
    dcc.Dropdown(id='x-var', options=stat_cols, value='wins'),
    dcc.Dropdown(id='y-var', options=stat_cols, value='losses'),
    dcc.Graph(id='scatter')
])

@app.callback(Output('scatter', 'figure'),
              Input('x-var', 'value'),
              Input('y-var', 'value'))
def update_graph(x_var, y_var):
    return px.scatter(df, x=x_var, y=y_var,
                      hover_name='teamFullName')

if __name__ == '__main__':
    app.run(debug=True)

### Shiny for Python {.unnumbered}

[Shiny for Python](https://shiny.posit.co/py/) is Posit's official port of R's Shiny framework to Python. If you are familiar with Shiny in R, the Python version uses a very similar reactive programming model. It is the newest of the three options but is actively developed and well supported.

In [ ]:
#| eval: false

# shiny_app.py
from shiny import App, ui
from shinywidgets import render_plotly, output_widget
import plotly.express as px
import pandas as pd
from nhlpy import NHLClient

client = NHLClient()
team_stats = client.stats.team_summary(
    start_season='20242025', end_season='20242025'
)
df = pd.DataFrame(team_stats)

exclude = ['teamFullName', 'teamId', 'seasonId', 'ties']
stat_cols = [c for c in df.columns if c not in exclude]

app_ui = ui.page_sidebar(
    ui.sidebar(
        ui.input_select(
            "x_var", "X Variable",
            choices=stat_cols, selected="wins"
        ),
        ui.input_select(
            "y_var", "Y Variable",
            choices=stat_cols, selected="losses"
        ),
    ),
    output_widget("scatter"),
    title="NHL Team Statistics Explorer",
)

def server(input, output, session):
    @render_plotly
    def scatter():
        return px.scatter(df, x=input.x_var(),
                          y=input.y_var(),
                          hover_name='teamFullName')

app = App(app_ui, server)

Each of these frameworks can be deployed to the web. Streamlit offers free hosting through [Streamlit Community Cloud](https://streamlit.io/cloud). Dash apps can be deployed on any web server or through Plotly's enterprise platform. Shiny for Python apps can be deployed to [shinyapps.io](https://www.shinyapps.io) or [Posit Connect](https://posit.co/products/enterprise/connect/). Choose the framework that best fits your workflow and the complexity of your application.

Regardless of which framework you choose, the value of an interactive dashboard lies in translating your analysis into a product that anyone can explore without writing a line of code. 
As NHL data continues to grow richer, particularly with the expansion of player-tracking data, dashboards like these will become an increasingly important way to communicate analytical findings.
The following lab gives you the opportunity to build an interactive dashboard. 
We encourage you to build a small dashboard using one of these frameworks to explore the metrics introduced earlier in this chapter, such as Corsi, Fenwick, or expected goals. 
This is great practice for presenting your work to a non-technical audience.

___

## Case Study: Pulling the goaltender

There is nothing in the rules of ice hockey that requires a team to have a certain number of wingers or defensemen on the ice, nor is a team even required to have a goaltender playing. Due to this, one strategy that a team can employ is to remove the goaltender in favor of an extra attacking skater. This is a high variance strategy, as having six skaters vs five creates an obvious offensive advantage, but of course not having a goaltender creates an obvious defensive disadvantage for the same team. Because of this variance, this strategy is usually only employed late in a game when a team is behind and trying to tie the score.

Traditionally in ice hockey, a team who is trailing by one goal might take the goaltender off of the ice in favor of an extra attacking skater in the last two minutes or so of the game. The idea is that if they can manage to tie the game, that a tie is better than a loss (and in some leagues like the NHL, the game will go to overtime and give them a chance to win), but that if they give up an empty-net goal, it doesn't really matter as losing by two goals is virtually the same as losing by one goal.

The success rates of this strategy are not high, but then again the likelihood of scoring an even-strength goal in any two minute period is also not high. In the NHL in the 2013-2014 through the 2021 seasons, the success rate of pulling the goaltender when a team is losing by one goal, where success is defined as winning the game or reaching overtime, varied between 9.7% in 2014-2015 to a high of 18.1% in the 2021 shortened season.

While the strategy of pulling the goaltender when down one goal in the waning minutes of a game has been quite well known and practiced over many decades, there is much debate about expanding the strategy. For example, we could use the same logic and pull the goaltender at some point when a team was down two goals. Across the same seven seasons in the previous paragraph, it was tried about 2500 times and the success rate as defined above was approximately 1%. In that same time frame, no NHL team achieved a successful outcome by pulling the goaltender when down by three or more goals. Specifically, the success rate of tying or winning a game when down by one, two, or three or more goals was 13.6%, 1%, and 0%, respectively, from 2013-14 to 2021-22.

We also find that there are differences in the success rate depending on the average amount of time that the goaltender has been pulled. For example, across those same eight seasons, teams that achieved success had approximately 11 more seconds of extra attacker time than those that were unsuccessful (1:43 to 1:34). These differences cannot be attributed to simply a difference in coaching strategy, as hockey is a fluid game and a team cannot wisely pull their goaltender when the puck is on their half of the ice lest the opposing team have quite a large advantage. Furthermore, a team who has pulled their goaltender may also be strategically forced to put him back into the game if there is a faceoff in the defensive half of the ice or in the neutral zone in the middle of the ice. Many of these visualizations are available on a beautifully created Tableau Public site by Meghan Hall available at [https://public.tableau.com/app/profile/meghanhall/viz/GoaliePullTimes2018-19/GoaliePullTimes18-19](https://public.tableau.com/app/profile/meghanhall/viz/GoaliePullTimes2018-19/GoaliePullTimes18-19).

At what point in a game that a trailing team should pull their goaltender is the subject of much debate. Much like 4th down decisions in football (refer back to that chapter if you need more information), those not directly employed by the teams tend to suggest that in-game strategy is far too conservative. While NHL teams traditionally for decades have waited for the last two minutes of a game to pull a goaltender, analysts not directly involved with teams suggest it should be done much earlier. In the 2018 paper "Pulling the Goalie: Hockey and Investment Implications", authors Clifford Asness and Aaron Brown [https://papers.ssrn.com/sol3/papers.cfm?abstract_id=3132563](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=3132563) suggest that teams should pull the goaltender with 6:10 left in the game if they are down one goal, and 13:00 left if down two goals. Rob Vollman in 2018 developed ThePullBot, a Twitter account created through statistical modeling, that suggests a team down one goal should pull the goaltender at about the 4:10 mark. This account is no longer active and Vollman is now the Director of Pro Analytics for the NHL's Los Angeles Kings. Perhaps not coincidentally, the Kings have become much more aggressive about when they pull the goaltender since he arrived.

It is easy to see how the strategy affects the current game: if a team turns a loss into a tie or win, that's great. However we know that about six times out of seven this is not the case, at least in the NHL. For example, how might the psyche of a team be affected if they lose a game by two goals instead of just one, and is that effect stronger if the team plays the next day as opposed to two or more days later? Or is there an equally strong uplifting effect if a team manages a tie after pulling a goaltender? Or if the team loses in overtime after putting all the effort into getting tied by pulling the goaltender? Is the effect different in the playoffs, where an overtime loss counts for nothing in the standings, as opposed to in the regular season where a loss in overtime counts as half of a win?

What kind of data would you need to study this effect, and how would you go about it?

___

## Exercises

1. The Florida Panthers won the 2024-25 Stanley Cup. Using the NHL API, retrieve skater statistics for the 2024-25 season and filter for Panthers players (team abbreviation `'FLA'`). Which Panther had the most points? Which had the highest plus-minus?

In [ ]:
#| eval: false
#| code-fold: true
#| code-summary: "Show solution"

client = NHLClient()
skaters = client.stats.skater_stats_summary(
    start_season='20242025',
    end_season='20242025',
    limit=-1
)
skaters_df = pd.DataFrame(skaters)
fla = skaters_df[skaters_df['teamAbbrevs'] == 'FLA'].copy()

top_pts = fla.sort_values('points', ascending=False).head(5)
print("Top scorers:\n",
      top_pts[['skaterFullName', 'points', 'goals', 'assists']])

top_pm = fla.sort_values('plusMinus', ascending=False).head(5)
print("\nTop plus-minus:\n",
      top_pm[['skaterFullName', 'plusMinus']])

For problems 2 and 3, use the `nhl_data_hockey_reference` dataset from the `isapy_data` module.

2. Find the correlation between Corsi percent and Fenwick percent by year using the entire dataset. Filter out all `NA` values on those variables.

In [ ]:
#| eval: false
#| code-fold: true
#| code-summary: "Show solution"

nhl_clean = nhl_data.dropna(subset=['cf_percent', 'ff_percent'])
corr_by_year = (nhl_clean
                .groupby('season')
                .apply(lambda x: x['cf_percent'].corr(x['ff_percent']))
                .reset_index(name='correlation'))
print(corr_by_year)

3. Recall in @fig-corsi-pos that we have to deal with data that has created two new positions for us, and it makes our graphic look a bit strange.

    * Explore the data for all players in that data set who have the position W. How many unique players are there?
    * Explore the data for all players in that data set who have the position F. How many unique players are there?
    * Is there a way to aggregate this data to allow for consistency with the previous existing data? What is your strategy to do it?
    * Re-create @fig-corsi-pos following your strategy to make it look nicer and more informative.

In [ ]:
#| eval: false
#| code-fold: true
#| code-summary: "Show solution"

nhl_data = isapy_data.load_data("nhl_data_hockey_reference")

w_players = nhl_data[nhl_data['pos'] == 'W']['player'].nunique()
f_players = nhl_data[nhl_data['pos'] == 'F']['player'].nunique()
print(f"Unique players with pos='W': {w_players}")
print(f"Unique players with pos='F': {f_players}")

# Strategy: map W -> LW (or could split), F -> C as a default
# A better approach: drop W and F rows, or reclassify based
# on the player's most common position in other seasons
nhl_clean = nhl_data[~nhl_data['pos'].isin(['W', 'F'])].copy()

nhl_tmp = (nhl_clean[nhl_clean['toi'] >= 200]
           .groupby(['season', 'pos'])
           .agg(avg_corsi=('cf_percent', 'mean'))
           .reset_index())

fig, ax = plt.subplots(figsize=(10, 6))
palette = sns.color_palette("Dark2")
for i, pos in enumerate(sorted(nhl_tmp['pos'].unique())):
    pos_data = nhl_tmp[nhl_tmp['pos'] == pos]
    ax.plot(pos_data['season'], pos_data['avg_corsi'],
            marker='o', label=pos, color=palette[i])
ax.set_xlabel('Season')
ax.set_ylabel('Average Corsi')
ax.set_xticks(range(2012, 2024))
ax.tick_params(axis='x', rotation=45)
ax.legend(title='Position')
plt.tight_layout()
plt.show()

4. Create a shot chart for the Vegas Golden Knights for their 2024-25 regular season games. Use the NHL API to fetch the schedule and play-by-play data, then plot all goals on an `NHLRink`. The Vegas Golden Knights team ID is 54.

In [ ]:
#| eval: false
#| code-fold: true
#| code-summary: "Show solution"

client = NHLClient()
vgk_schedule = client.schedule.team_season_schedule('VGK', '20242025')
vgk_games = [g for g in vgk_schedule.get('games', []) if g.get('gameType') == 2]

vgk_plays = []
for game in vgk_games:
    pbp = client.game_center.play_by_play(game['id'])
    for play in pbp.get('plays', []):
        details = play.get('details', {})
        if details.get('eventOwnerTeamId') == 54 and play.get('typeDescKey') == 'goal':
            vgk_plays.append({
                'x': details['xCoord'],
                'y': details['yCoord'],
                'shot_type': details.get('shotType'),
            })
    time.sleep(0.5)

vgk_goals = pd.DataFrame(vgk_plays)

color_updates = {
    'plot_background': '#ffffff', 'boards': '#000000',
    'ozone_ice': '#ffffff', 'nzone_ice': '#ffffff', 'dzone_ice': '#ffffff',
    'center_line': '#c8102e', 'zone_line': '#0033a0', 'goal_line': '#c8102e',
}
rink = NHLRink(color_updates=color_updates)
ax = rink.draw(display_range='full')
ax.scatter(vgk_goals['x'], vgk_goals['y'],
           alpha=0.6, s=30, color='#B4975A', zorder=100)
plt.show()

5. Color the Vegas Golden Knights goals from Exercise 4 by shot type. Which shot type produces the most goals?

In [ ]:
#| eval: false
#| code-fold: true
#| code-summary: "Show solution"

# Using vgk_goals from Exercise 4
rink = NHLRink(color_updates=color_updates)
ax = rink.draw(display_range='full')

shot_types = vgk_goals['shot_type'].dropna().unique()
palette = sns.color_palette("Set2", len(shot_types))
for i, st in enumerate(shot_types):
    subset = vgk_goals[vgk_goals['shot_type'] == st]
    ax.scatter(subset['x'], subset['y'],
               alpha=0.7, s=30, color=palette[i],
               label=st, zorder=100)
ax.legend(title='Shot Type', loc='lower center',
          bbox_to_anchor=(0.5, -0.15), ncol=3)
plt.show()

print(vgk_goals['shot_type'].value_counts())

6. Using the `team_stats_df` data fetched earlier in this chapter from the NHL API, create an interactive scatter plot of any two team statistics using `plotly`. Add team names as hover labels.

In [ ]:
#| eval: false
#| code-fold: true
#| code-summary: "Show solution"

import plotly.express as px

fig = px.scatter(
    team_stats_df,
    x='goalsFor',
    y='goalsAgainst',
    hover_name='teamFullName',
    labels={
        'goalsFor': 'Goals For',
        'goalsAgainst': 'Goals Against'
    },
    title='NHL Teams: Goals For vs Goals Against (2024-25)'
)
fig.show()